# Delta Lake MERGE Implementation — Week 7 Assignment
### Celebal Technologies Excellence Internship

**Objective:** Perform incremental data processing using Delta Lake — load the provided
Superstore dataset, clean it, simulate an incremental feed on top of it, and apply
`MERGE INTO` to update existing records and insert new ones.

**Dataset:** `Sample - Superstore.csv` (the resource given with this assignment) — 9,994
order line items with customer, product, and sales/profit details.

**Environment:** Databricks Community Edition (Runtime with Delta Lake pre-installed).

---


## Step 1: Load Dataset into a Delta Table



In [0]:
master_path = "/Workspace/delta-lake-assignment/superstore_master.csv"
incremental_path = "/Workspace/delta-lake-assignment/superstore_incremental.csv"

df_master_raw = (spark.read
                  .option("header", True)
                  .option("inferSchema", True)
                  .csv(master_path))

print("Rows loaded from master file:", df_master_raw.count())
df_master_raw.printSchema()
df_master_raw.show(5, truncate=False)

Rows loaded from master file: 9994
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+--

In [0]:
from pyspark.sql.functions import col

# Rename columns by replacing spaces, hyphens and slashes with underscores
df_master = df_master_raw.select(
    *[
        col(c).alias(
            c.replace(" ", "_")
             .replace("-", "_")
             .replace("/", "_")
        )
        for c in df_master_raw.columns
    ]
)

print("Schema after renaming columns")

df_master.printSchema()

display(df_master.limit(5))

Schema after renaming columns
root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


In [0]:
# Drop the table if it already exists
spark.sql("DROP TABLE IF EXISTS superstore_raw")

(
    df_master.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("superstore_raw")
)

display(spark.sql("SELECT * FROM superstore_raw LIMIT 20"))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0,34.47


In [0]:
spark.table("superstore_raw").printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



Screenshot the schema + table preview above into `screenshots/data_loading/`.

## Step 2: Basic Cleaning (Handle Nulls, Remove Duplicates)


Check the dataset for duplicate and missing values. Demonstrate the cleaning process using a small synthetic dataset, then apply the same cleaning logic to the original dataset and save it as the master Delta table.

In [0]:
from pyspark.sql import functions as F

# ----- Build a small synthetic "dirty" dataset -----

# Create duplicate rows
dirty_duplicates = df_master.limit(5)

# Create rows with null values
dirty_nulls = (
    df_master.limit(5)
    .withColumn("City", F.lit(None).cast("string"))
    .withColumn("Postal_Code", F.lit(None).cast("int"))
    .withColumn("Row_ID", F.col("Row_ID") + 100000)
)

# Combine original + duplicates + null rows
df_test_dirty = (
    df_master
    .unionByName(dirty_duplicates)
    .unionByName(dirty_nulls)
)

print("Row count with synthetic dirty rows:", df_test_dirty.count())

print(
    "Duplicate rows:",
    df_test_dirty.count() - df_test_dirty.dropDuplicates().count()
)

print(
    "Rows with null values:",
    df_test_dirty.filter(
        F.col("City").isNull() |
        F.col("Postal_Code").isNull()
    ).count()
)

Row count with synthetic dirty rows: 10004
Duplicate rows: 5
Rows with null values: 5


In [0]:
# Remove duplicates
df_clean_test = df_test_dirty.dropDuplicates()

# Fill null values
df_clean_test = df_clean_test.fillna({
    "City": "Unknown",
    "Postal_Code": 0
})

print("Row count after cleaning:", df_clean_test.count())

print(
    "Returned to original dataset?",
    df_clean_test.count() == df_master.count()
)

display(df_clean_test.limit(5))

Row count after cleaning: 9999
Returned to original dataset? False


Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"""Stur-D-Stor Shelving, Vertical 5-Shelf: 72""""H x 36""""W x 18 1/2""""D""",665.88,6,0,13.3176
21,CA-2014-143336,2014-08-27,2014-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109,West,OFF-BI-10002215,Office Supplies,Binders,"Wilson Jones Hanging View Binder, White, 1""""",22.72,4,0.2,7.384
53,CA-2015-115742,2015-04-18,2015-04-22,Standard Class,DP-13000,Darren Powers,Consumer,United States,New Albany,Indiana,47150,Central,FUR-CH-10003061,Furniture,Chairs,"Global Leather Task Chair, Black",89.99,1,0,17.0981
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0,165.3813


In [0]:
# Clean the original dataset
df_clean = df_master.dropDuplicates()

df_clean = df_clean.fillna({
    "City": "Unknown",
    "Postal_Code": 0
})

print("Final cleaned row count:", df_clean.count())

# Recreate the Delta table
spark.sql("DROP TABLE IF EXISTS superstore_master")

(
    df_clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("superstore_master")
)

print("superstore_master Delta table created.")

Final cleaned row count: 9994
superstore_master Delta table created.


In [0]:
spark.table("superstore_master").printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



## Step 3: Create the Incremental Dataset

Load the incremental dataset, standardize the column names, and save it as a Delta table to simulate new incoming records.

In [0]:
df_incremental_raw = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(incremental_path)
)

print("Incoming incremental records:", df_incremental_raw.count())

df_incremental_raw.printSchema()

display(df_incremental_raw.limit(5))

Incoming incremental records: 30
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1687,CA-2017-149489,2017-04-24,2017-04-27,First Class,DK-12835,Damala Kotsonis,Corporate,United States,Philadelphia,Pennsylvania,19143,East,OFF-BI-10002414,Office Supplies,Binders,GBC ProClick Spines for 32-Hole Punch,7.8939,3.0,0.8,-5.131
4897,CA-2016-135776,2016-12-23,2016-12-30,First Class,EH-13765,Edward Hooks,Corporate,United States,Seattle,Washington,98103,West,OFF-ST-10003470,Office Supplies,Storage,"Tennsco Snap-Together Open Shelving Units, Starter Sets and Add-On Units",1173.816,5.0,0.1,58.6908
6070,CA-2017-166695,2017-05-20,2017-05-24,First Class,CC-12430,Chuck Clark,Home Office,United States,Lakewood,California,90712,West,TEC-MA-10003176,Technology,Machines,Okidata B400 Printer,360.36,2.0,0.3,-54.054
2475,CA-2016-114972,2016-11-03,2016-11-06,First Class,PF-19225,Phillip Flathmann,Consumer,United States,Los Angeles,California,90032,West,TEC-AC-10000682,Technology,Accessories,Kensington K72356US Mouse-in-a-Box USB Desktop Mouse,87.0975,6.0,0.1,4.3549
8460,CA-2014-126200,2014-08-25,2014-08-29,First Class,JE-15715,Joe Elijah,Consumer,United States,Houston,Texas,77070,Central,OFF-BI-10002225,Office Supplies,Binders,"""Square Ring Data Binders, Rigid 75 Pt. Covers, 11"""" x 14-7/8""""""",13.0032,4.0,0.8,-8.4521


In [0]:
from pyspark.sql.functions import col

df_incremental = df_incremental_raw.select(
    *[
        col(c).alias(
            c.replace(" ", "_")
             .replace("-", "_")
             .replace("/", "_")
        )
        for c in df_incremental_raw.columns
    ]
)

print("Schema after renaming incremental columns")

df_incremental.printSchema()

display(df_incremental.limit(5))

Schema after renaming incremental columns
root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1687,CA-2017-149489,2017-04-24,2017-04-27,First Class,DK-12835,Damala Kotsonis,Corporate,United States,Philadelphia,Pennsylvania,19143,East,OFF-BI-10002414,Office Supplies,Binders,GBC ProClick Spines for 32-Hole Punch,7.8939,3.0,0.8,-5.131
4897,CA-2016-135776,2016-12-23,2016-12-30,First Class,EH-13765,Edward Hooks,Corporate,United States,Seattle,Washington,98103,West,OFF-ST-10003470,Office Supplies,Storage,"Tennsco Snap-Together Open Shelving Units, Starter Sets and Add-On Units",1173.816,5.0,0.1,58.6908
6070,CA-2017-166695,2017-05-20,2017-05-24,First Class,CC-12430,Chuck Clark,Home Office,United States,Lakewood,California,90712,West,TEC-MA-10003176,Technology,Machines,Okidata B400 Printer,360.36,2.0,0.3,-54.054
2475,CA-2016-114972,2016-11-03,2016-11-06,First Class,PF-19225,Phillip Flathmann,Consumer,United States,Los Angeles,California,90032,West,TEC-AC-10000682,Technology,Accessories,Kensington K72356US Mouse-in-a-Box USB Desktop Mouse,87.0975,6.0,0.1,4.3549
8460,CA-2014-126200,2014-08-25,2014-08-29,First Class,JE-15715,Joe Elijah,Consumer,United States,Houston,Texas,77070,Central,OFF-BI-10002225,Office Supplies,Binders,"""Square Ring Data Binders, Rigid 75 Pt. Covers, 11"""" x 14-7/8""""""",13.0032,4.0,0.8,-8.4521


In [0]:
# Recreate incremental Delta table
spark.sql("DROP TABLE IF EXISTS superstore_incremental")

(
    df_incremental.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("superstore_incremental")
)

print("superstore_incremental Delta table created.")

superstore_incremental Delta table created.


In [0]:
spark.table("superstore_incremental").printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



## Step 4a: MERGE — Update Existing, Insert New (on `Row ID`)

`Row ID` is the natural primary key for this dataset (one row = one order line item), so the merge matches on that.

In [0]:
spark.table("superstore_master").printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
spark.table("superstore_incremental").printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
from delta.tables import DeltaTable

# Load Delta table
delta_superstore = DeltaTable.forName(spark, "superstore_master")

# Perform MERGE
(
    delta_superstore.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdate(
        set={
            "Order_ID": "source.Order_ID",
            "Order_Date": "source.Order_Date",
            "Ship_Date": "source.Ship_Date",
            "Ship_Mode": "source.Ship_Mode",
            "Customer_ID": "source.Customer_ID",
            "Customer_Name": "source.Customer_Name",
            "Segment": "source.Segment",
            "Country": "source.Country",
            "City": "source.City",
            "State": "source.State",
            "Postal_Code": "source.Postal_Code",
            "Region": "source.Region",
            "Product_ID": "source.Product_ID",
            "Category": "source.Category",
            "Sub_Category": "source.Sub_Category",
            "Product_Name": "source.Product_Name",
            "Sales": "source.Sales",
            "Quantity": "source.Quantity",
            "Discount": "source.Discount",
            "Profit": "source.Profit"
        }
    )
    .whenNotMatchedInsert(
        values={
            "Row_ID": "source.Row_ID",
            "Order_ID": "source.Order_ID",
            "Order_Date": "source.Order_Date",
            "Ship_Date": "source.Ship_Date",
            "Ship_Mode": "source.Ship_Mode",
            "Customer_ID": "source.Customer_ID",
            "Customer_Name": "source.Customer_Name",
            "Segment": "source.Segment",
            "Country": "source.Country",
            "City": "source.City",
            "State": "source.State",
            "Postal_Code": "source.Postal_Code",
            "Region": "source.Region",
            "Product_ID": "source.Product_ID",
            "Category": "source.Category",
            "Sub_Category": "source.Sub_Category",
            "Product_Name": "source.Product_Name",
            "Sales": "source.Sales",
            "Quantity": "source.Quantity",
            "Discount": "source.Discount",
            "Profit": "source.Profit"
        }
    )
    .execute()
)

print("MERGE completed successfully.")

display(
    spark.sql("""
        SELECT *
        FROM superstore_master
        ORDER BY Row_ID DESC
        LIMIT 15
    """)
)

MERGE completed successfully.


Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
10004,CA-2017-900009,2017-01-05,2017-01-09,Second Class,LP-17080,Liz Pelletier,Consumer,United States,San Francisco,California,94110,West,OFF-BI-10000069,Office Supplies,Binders,"GBC Prepunched Paper, 19-Hole, for Binding Systems, 24-lb",75.6367,5.0,0.1,7.5637
10003,CA-2017-900008,2017-01-05,2017-01-09,Standard Class,SS-20140,Saphhira Shifley,Corporate,United States,Columbus,Georgia,31907,South,OFF-PA-10000673,Office Supplies,Paper,"Post-it Important Message Note Pad, Neon Colors, 50 Sheets/Pad",60.7743,3.0,0.0,12.1549
10002,CA-2017-900007,2017-01-05,2017-01-09,Second Class,PF-19165,Philip Fox,Consumer,United States,Sandy Springs,Georgia,30328,South,FUR-BO-10004695,Furniture,Bookcases,O'Sullivan 2-Door Barrister Bookcase in Odessa Pine,1302.3764,3.0,0.0,260.4753
10001,CA-2017-900006,2017-01-05,2017-01-09,Standard Class,FH-14350,Fred Harton,Consumer,United States,Atlanta,Georgia,30318,South,OFF-PA-10003893,Office Supplies,Paper,Xerox 1962,21.0401,3.0,0.1,2.104
10000,CA-2017-900005,2017-01-05,2017-01-09,Standard Class,PB-19150,Philip Brown,Consumer,United States,Los Angeles,California,90004,West,FUR-FU-10004845,Furniture,Furnishings,"Deflect-o EconoMat Nonstudded, No Bevel Mat",439.7817,2.0,0.1,43.9782
9999,CA-2017-900004,2017-01-05,2017-01-09,Same Day,AH-10120,Adrian Hane,Home Office,United States,Tucson,Arizona,85705,West,FUR-CH-10002372,Furniture,Chairs,Office Star - Ergonomically Designed Knee Chair,225.3827,5.0,0.1,22.5383
9998,CA-2017-900003,2017-01-05,2017-01-09,Standard Class,BW-11200,Ben Wallace,Consumer,United States,Skokie,Illinois,60076,Central,OFF-BI-10000309,Office Supplies,Binders,"""GBC Twin Loop Wire Binding Elements, 9/16"""" Spine","Black""",13.4658,5.0,0.1
9997,CA-2017-900002,2017-01-05,2017-01-09,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002033,Technology,Phones,Konftel 250 Conference phone - Charcoal black,861.8558,3.0,0.0,172.3712
9996,CA-2017-900001,2017-01-05,2017-01-09,Second Class,GA-14515,George Ashbrook,Consumer,United States,New York City,New York,10011,East,FUR-FU-10003601,Furniture,Furnishings,"Deflect-o RollaMat Studded, Beveled Mat for Medium Pile Carpeting",281.1094,5.0,0.1,28.1109
9995,CA-2017-900000,2017-01-05,2017-01-09,Standard Class,JM-15865,John Murray,Consumer,United States,Arlington,Virginia,22204,South,OFF-AP-10003281,Office Supplies,Appliances,Acco 6 Outlet Guardian Standard Surge Suppressor,38.5325,4.0,0.2,0.0


## Step 4b: MERGE with History (SCD Type 2) on the 20 Updated Orders

Implement Slowly Changing Dimension (SCD Type 2) to preserve historical data by maintaining record versions using effective dates and the current status flag.


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Initialize SCD2 table
df_scd2_init = (
    df_clean
    .withColumn("effective_date", F.current_date())
    .withColumn("end_date", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
)

spark.sql("DROP TABLE IF EXISTS superstore_master_scd2")

(
    df_scd2_init.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("superstore_master_scd2")
)

print(
    "superstore_master_scd2 initialized with",
    df_scd2_init.count(),
    "rows."
)

scd2_table = DeltaTable.forName(spark, "superstore_master_scd2")

superstore_master_scd2 initialized with 9994 rows.


In [0]:
staged_updates = (
    df_incremental.alias("source")
    .join(
        spark.table("superstore_master_scd2")
             .filter("is_current = true")
             .alias("target"),
        "Row_ID"
    )
    .where(
        (F.col("source.Ship_Mode") != F.col("target.Ship_Mode")) |
        (F.col("source.Quantity") != F.col("target.Quantity")) |
        (F.col("source.Discount") != F.col("target.Discount")) |
        (F.col("source.Profit") != F.col("target.Profit"))
    )
    .select("source.*")
)

print(
    "Records requiring SCD2 update:",
    staged_updates.count()
)

Records requiring SCD2 update: 20


In [0]:
# Close old current records
(
    scd2_table.alias("target")
    .merge(
        staged_updates.alias("source"),
        "target.Row_ID = source.Row_ID AND target.is_current = true"
    )
    .whenMatchedUpdate(
        set={
            "end_date": "current_date()",
            "is_current": "false"
        }
    )
    .execute()
)

print("Existing records updated successfully.")

Existing records updated successfully.


In [0]:
new_or_changed = (
    df_incremental.alias("source")
    .join(
        spark.table("superstore_master_scd2")
             .filter("is_current = true"),
        "Row_ID",
        "left_anti"
    )
)

df_new_versions = (
    new_or_changed
    .withColumn("effective_date", F.current_date())
    .withColumn("end_date", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
)

(
    df_new_versions.write
        .format("delta")
        .mode("append")
        .saveAsTable("superstore_master_scd2")
)

print("New versions inserted successfully.")

New versions inserted successfully.


In [0]:
print("SCD2 MERGE completed.")

display(
    spark.sql("""
        SELECT
            Row_ID,
            Ship_Mode,
            Quantity,
            Discount,
            Profit,
            effective_date,
            end_date,
            is_current
        FROM superstore_master_scd2
        WHERE Row_ID IN (
            SELECT Row_ID
            FROM superstore_incremental
            WHERE Row_ID <= 9994
        )
        ORDER BY Row_ID, effective_date
        LIMIT 20
    """)
)

SCD2 MERGE completed.


Row_ID,Ship_Mode,Quantity,Discount,Profit,effective_date,end_date,is_current
421,Standard Class,1,0,7.015,2026-07-13,2026-07-13,false
421,First Class,2.0,0.1,0.8006,2026-07-13,null,true
857,First Class,11.0,0.1,5.3707,2026-07-13,null,true
857,Standard Class,10,0,26.598,2026-07-13,2026-07-13,false
1687,First Class,3.0,0.8,-5.131,2026-07-13,null,true
1687,First Class,2,0.7,-5.7638,2026-07-13,2026-07-13,false
2475,First Class,6.0,0.1,4.3549,2026-07-13,null,true
2475,First Class,5,0,29.0325,2026-07-13,2026-07-13,false
4092,Standard Class,3,0.2,4.6644,2026-07-13,2026-07-13,false
4092,First Class,4.0,0.3,-2.2604,2026-07-13,null,true


## Step 5: Validate Results (Row Counts, Duplicates)

Validate the merged data by checking the total row count, unique Row_ID values, and verifying that updated and newly inserted records are present.

In [0]:
total_rows = spark.table("superstore_master").count()

distinct_ids = (
    spark.table("superstore_master")
         .select("Row_ID")
         .distinct()
         .count()
)

print(f"Total rows in superstore_master : {total_rows}")
print(f"Distinct Row_ID values          : {distinct_ids}")
print(f"Expected total (9994 + 10 new)  : {9994 + 10}")

print(
    "Duplicates present?",
    "YES - investigate" if total_rows != distinct_ids else "No duplicates ✅"
)

display(
    spark.sql("""
        SELECT
            Row_ID,
            Ship_Mode,
            Quantity,
            Discount,
            Profit
        FROM superstore_master
        WHERE Row_ID IN (1687,4897,6070,9995,9996,10004)
        ORDER BY Row_ID
    """)
)

Total rows in superstore_master : 10004
Distinct Row_ID values          : 10004
Expected total (9994 + 10 new)  : 10004
Duplicates present? No duplicates ✅


Row_ID,Ship_Mode,Quantity,Discount,Profit
1687,First Class,3.0,0.8,-5.131
4897,First Class,5.0,0.1,58.6908
6070,First Class,2.0,0.3,-54.054
9995,Standard Class,4.0,0.2,0.0
9996,Second Class,5.0,0.1,28.1109
10004,Second Class,5.0,0.1,7.5637


## Step 6: Final Dataset and Summary


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM superstore_master
        ORDER BY Row_ID
        LIMIT 30
    """)
)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0,34.47


In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY superstore_master
    """)
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-07-13T16:10:43.000Z,76040129367505,siganithin@gmail.com,MERGE,"Map(predicate -> [""(Row_ID#18072 = Row_ID#17213)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2835168391517359),37617f3f-af6a-4d78-9227-2fe10df5e3a0,0713-155229-oxmfh14n-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 9104, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 20, executionTimeMs -> 6672, materializeSourceTimeMs -> 554, numTargetRowsInserted -> 10, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3059, numTargetRowsUpdated -> 20, numOutputRows -> 30, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 30, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2932)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-13T16:08:12.000Z,76040129367505,siganithin@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2835168391517359),ab16c2bc-446d-4845-878a-ce5db22d575a,0713-155229-oxmfh14n-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 9994, numOutputBytes -> 335144)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


---
### Summary

- Loaded the Superstore dataset into a Delta table.
- Renamed column names to a Delta-compatible format using underscores.
- Verified and cleaned the data by removing duplicates and handling null values.
- Created an incremental dataset containing updated and new records.
- Performed Delta Lake MERGE to update existing rows and insert new rows.
- Implemented SCD Type 2 to preserve historical records using `effective_date`, `end_date`, and `is_current`.
- Validated the final dataset by checking row counts and duplicate Row_ID values.
- Verified Delta transaction history using `DESCRIBE HISTORY`.